<a href="https://colab.research.google.com/github/hursoo/big_k-modern_1/blob/main/gb_052_gb_feature_dtm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1.개요

In [1]:
# 구글 드라이브 마운트

from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
from collections import Counter   # 글자 수 계산에 유용한 패키지

# 2.개벽의 특성 벡터 추출 1: 고빈도 단어
- 빈도 단위는 tfidf 값으로,
- 개수는 고빈도 단어 50개로 설정

In [3]:
# 경로 지정

file_path = '/content/drive/MyDrive/big_km_history01/'

In [4]:
# 통합 데이터 불러오기
gb_df = pd.read_excel(file_path + 'result/gb_data_2(doc,1g2g,wn_cls).xlsx') ###
print(gb_df.shape)
gb_df.head(3)

(6802, 9)


,doc_id,doc_raw,doc_split_12gram,r_no,title,w_new,ho_no,grid_1,wn_cls
0,1,創刊辭 强者도 부르짖고 弱者도 부르짖으며 優者도 부르짖고 劣者도 부르짖도다 東西南北...,창간 辭 강자 약자 優者 劣者 동서 남북 사해 팔방 소리 소리 판단 좌우 間 다수 ...,1,創刊辭,uk01,1,01q,0
1,2,哲人은 말하되 多數 人民의 聲은 곳 神의 聲이라 하엿나니 神은 스스로 要求가 없는지...,哲人 다수 인민 요구 인민 소리 요구 발표 갈앙 인민 소리 갈앙 다수 인민 갈앙 요...,1,創刊辭,uk01,1,01q,0
2,3,世界를 알라 사람은 天使도 안이며 野獸도 안이오 오즉 사람일 뿐이로다 이만치 進化된...,세계 사람 야수 사람 진화 진화 지식 진화 도덕 동물 세계 천당 지옥 세계 진화 국...,2,世界를 알라,uk01,1,01q,0


In [5]:
# 문서-단어 행렬

import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def get_dtm(df, col_name, stopw, rank_n): # rank_n : 고빈도 단어 n 순위까지
    '''
    # 문서-단어 행렬(dtm) 산출 함수
    '''
    # 단어 종류 모두 벡터화. 1음절 이상
    tv = TfidfVectorizer(ngram_range=(1,1), stop_words=stopw) ## 두 글자 이상
    dtm = tv.fit_transform(df[col_name])

    # tfidf 합계를 사용해 고빈도 단어 추출 (희소 행렬로 작업)
    term_sums = np.array(dtm.sum(axis=0)).flatten()
    highword_indices = term_sums.argsort()[-rank_n:][::-1]  # 상위 rank_n 개 단어 인덱스
    highword_list = [tv.get_feature_names_out()[i] for i in highword_indices]

    # 고빈도 단어만 포함된 희소 행렬 생성
    feature_dtm = dtm[:, highword_indices]
    feature_df = pd.DataFrame(feature_dtm.toarray(), columns=highword_list, index=df.index)

    return feature_df

In [6]:
# 함수 실행

stopw = ['문제', '금일', '관계']  # 제외할 단어
dtm_gb_df = get_dtm(gb_df, 'doc_split_12gram', stopw, 50)
dtm_gb_df

,사람,사회,朝鮮,생활,사상,운동,自己,세계,민족,主義,...,혁명,발달,현재,사업,意識,문명,목적,노동,생산,방법
0,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.031663,0.00000,0.0,0.0,0.0,0.0,0.000000,0.077667,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.150628,0.07488,0.0,0.0,0.0,0.0,0.000000,0.277109,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.468555,0.00000,0.0,0.0,0.0,0.0,0.197656,0.670442,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.070227,0.00000,0.0,0.0,0.0,0.0,0.000000,0.430650,0.090315,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6797,0.068947,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6798,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6799,0.000000,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6800,0.082986,0.00000,0.0,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [7]:
# 단어 빈도 확인

dtm_gb_df.sum()

,0
사람,201.808757
사회,173.281553
朝鮮,152.025952
생활,142.297271
사상,131.171816
운동,109.253477
自己,108.402774
세계,107.769487
민족,105.871938
主義,103.643982


In [8]:
hw50 = print(dtm_gb_df.columns.tolist())
hw50

['사람', '사회', '朝鮮', '생활', '사상', '운동', '自己', '세계', '민족', '主義', '계급', '시대', '경제', '朝鮮_人', '정신', '인류', '자유', '단체', '교육', '정치', '민중', '일본', '필요', '개인', '도덕', '자연', '국가', '인간', '의미', '理想', '종교', '중국', '문화', '현상', '일반', '조직', '생명', '역사', '농민', '過去', '혁명', '발달', '현재', '사업', '意識', '문명', '목적', '노동', '생산', '방법']


In [9]:
dtm_gb_df.loc[6800,:].values

array([0.08298621, 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.12241703, 0.        , 0.        , 0.        ,
       0.        , 0.12220194, 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ,
       0.        , 0.        , 0.        , 0.        , 0.        ])

# 3.개벽의 특성 벡터 추출 2: 토픽
- 토픽 4개로 설정
- 토픽 개수, 알파값, 베타값, seed 값은 다양하게 설정할 수 있으며, 그 과정은 이후 작업에서 본격적으로 살펴볼 것이다.
- 다만, 여기서는 '고빈도 단어' 특성 추출과 비교할 목적으로, 나중에 최종 확정된 토픽 결과물을 산출하는 데 사용한 설정값을 사용한다.

In [10]:
# tomotopy_colab_setup.ipynb

# ✅ 자동 설치 및 재시작 유도
!pip install -q tomotopy==0.13.0

import os
import IPython

# numpy가 이미 import되었으면 런타임 재시작 필요
if 'numpy' in globals():
    print("📌 런타임 재시작이 필요합니다. 이 셀 실행 후 다시 실행해주세요.")
    IPython.display.display(IPython.display.Javascript('''google.colab.kernel.restart()'''))
else:
    import tomotopy
    print("✅ tomotopy 로드 성공")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 81.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.
✅ tomotopy 로드 성공


In [11]:
import sys
import os, re
import pandas as pd
import numpy as np
import random
import warnings

from tomotopy import DMRModel
from tomotopy import TermWeight
from tomotopy import utils
import tomotopy as tp

In [12]:
# 1) OpenMP 스레드 수 제한
os.environ["OMP_NUM_THREADS"] = "1"

# 2) 파이썬 자체 random, numpy 랜덤 시드 고정
def set_global_seeds(seed_value=1000):
    random.seed(seed_value)
    np.random.seed(seed_value)

# 3) `a_data`를 `metadata`로 변환하는 함수
def transform_a_data_to_metadata(misc: dict):
    return {'metadata': str(misc['a_data'])}

def run_dmr_model(gridL, lineL, num_topics=4, seed=1000, iterations=500, alpha=0.1, eta=0.01): ###
    """
    DMR 모델 실행 및 메타데이터 저장.

    Parameters:
        gridL (list): 메타데이터 리스트 (예: 발행 연도 등).
        lineL (list): 텍스트 데이터 리스트.
        num_topics (int): 토픽 개수. 기본값은 12.
        seed (int): 랜덤 시드. 기본값은 1000.
        iterations (int): 학습 반복 횟수. 기본값은 500.
        alpha (float): 알파값 (문서-토픽 분포 하이퍼파라미터).
        eta (float): 베타값 (토픽-단어 분포 하이퍼파라미터).

    Returns:
        model (DMRModel): 학습된 DMR 모델.
        topics (list): 각 토픽별 상위 단어 리스트.
    """
    # 파이썬 랜덤, numpy 랜덤 시드 고정
    set_global_seeds(seed)

    print(f"\nTraining DMR Model with {num_topics} topics, alpha={alpha}, eta={eta}...")

    # DMR 모델 초기화(tomotopy 내부 시드 설정)
    model = DMRModel(
        k=num_topics,
        seed=seed,
        tw=TermWeight.ONE,
        alpha=alpha,
        eta=eta
    )
    corpus = utils.Corpus()

    # 코퍼스에 문서 추가
    for grid, line in zip(gridL, lineL):
        tokens = line.strip().split()
        corpus.add_doc(tokens, a_data=grid)

    # 모델에 코퍼스 추가 (메타데이터 변환 포함)
    model.add_corpus(corpus, transform=transform_a_data_to_metadata)

    # 학습
    for i in range(0, iterations, 20):  # 20단위로 학습 반복
        model.train(20)
        print(f"Iteration: {i + 20}\tLog-likelihood: {model.ll_per_word:.4f}")

    # 토픽 결과 저장 및 출력
    topics = [model.get_topic_words(i, top_n=20) for i in range(model.k)]
    for idx, topic in enumerate(topics):
        print(f"Topic {idx}: {[word[0] for word in topic]}")

    return model, topics

In [13]:
# 사용 예시
gridL = gb_df['grid_1'].to_list()  # 메타데이터
lineL = gb_df['doc_split_12gram'].to_list()  # 텍스트 데이터

model, topics = run_dmr_model(
    gridL, lineL, num_topics=4, seed=103, iterations=1000, alpha=0.05, eta=0.1  # alpha=0.05, eta=0.1   ## seed를 '103'로 지정
)


Training DMR Model with 4 topics, alpha=0.05, eta=0.1...


/tmp/ipython-input-12-2753150319.py:55: RuntimeWarning: The training result may differ even with fixed seed if `workers` != 1.
  model.train(20)


Iteration: 20	Log-likelihood: -7.7975
Iteration: 40	Log-likelihood: -7.6460
Iteration: 60	Log-likelihood: -7.6130
Iteration: 80	Log-likelihood: -7.5978
Iteration: 100	Log-likelihood: -7.5915
Iteration: 120	Log-likelihood: -7.5872
Iteration: 140	Log-likelihood: -7.5865
Iteration: 160	Log-likelihood: -7.5873
Iteration: 180	Log-likelihood: -7.5842
Iteration: 200	Log-likelihood: -7.5803
Iteration: 220	Log-likelihood: -7.5819
Iteration: 240	Log-likelihood: -7.5827
Iteration: 260	Log-likelihood: -7.5863
Iteration: 280	Log-likelihood: -7.5848
Iteration: 300	Log-likelihood: -7.5865
Iteration: 320	Log-likelihood: -7.5848
Iteration: 340	Log-likelihood: -7.5870
Iteration: 360	Log-likelihood: -7.5850
Iteration: 380	Log-likelihood: -7.5839
Iteration: 400	Log-likelihood: -7.5853
Iteration: 420	Log-likelihood: -7.5869
Iteration: 440	Log-likelihood: -7.5836
Iteration: 460	Log-likelihood: -7.5842
Iteration: 480	Log-likelihood: -7.5828
Iteration: 500	Log-likelihood: -7.5836
Iteration: 520	Log-likelihood

In [14]:
# 문서별 토픽 구성 추출 및 DataFrame 생성 ---

# 토픽의 개수 확인
num_topics = model.k
topic_columns = [f'T{i}' for i in range(num_topics)]

# 결과를 저장할 리스트
doc_topic_distributions = []
document_indices = []

for i, doc in enumerate(model.docs):
    # doc.get_topic_dist()를 사용하여 해당 문서의 토픽 확률 분포를 가져옵니다.
    # 각 요소는 특정 토픽에 대한 문서의 확률입니다.
    topic_dist = doc.get_topic_dist()
    doc_topic_distributions.append(topic_dist)
    document_indices.append(f'{i}')

# NumPy 배열로 변환 후 DataFrame 생성
dt_df = pd.DataFrame(doc_topic_distributions, columns=topic_columns, index=document_indices)
dt_df

,T0,T1,T2,T3
0,0.3471654,0.63650817,0.013937129,0.0023892485
1,0.0014898471,0.96314937,0.034532856,0.0008279433
2,0.003809536,0.9817241,0.012349318,0.0021170494
3,0.003959916,0.9810026,0.012836803,0.002200619
4,0.0026870652,0.9067506,0.03549677,0.05506557
...,...,...,...,...
6797,0.17904392,0.0037502071,0.7729441,0.04426182
6798,0.5305792,0.005225956,0.4587314,0.005463444
6799,0.80033934,0.0042665615,0.19093375,0.004460451
6800,0.7801599,0.0046977764,0.21023113,0.004911262


# The End of Notes